# 04 — Perception + Qwen reasoner sandbox

Quick experiments with existing `s2r.models` adapters (YOLO / Qwen-VL / Qwen reasoner) on benchmark instructions.

In [ ]:
from pathlib import Path
import sys, time
sys.path.insert(0, str(Path.cwd() / "_lib"))
from bootstrap import setup
ROOT = setup()

from s2r.core.config import load_config
from s2r.experiments.benchmark import load_tasks
from s2r.models.registry import build_detector, build_reasoner, build_vlm
from s2r.models.yolo_detector import encode_image_stub

cfg = load_config(ROOT / "config" / "default.yaml")
# force mock for laptop experimentation; set false on GPU hosts
cfg["models"]["detector"]["mock"] = True
cfg["models"]["vlm"]["mock"] = True
cfg["models"]["reasoner"]["mock"] = True

det = build_detector(cfg)
vlm = build_vlm(cfg)
reasoner = build_reasoner(cfg)
tasks = load_tasks()
img = encode_image_stub()

In [ ]:
import pandas as pd

rows = []
for t in tasks:
    t0 = time.perf_counter()
    perc = det.infer(img, t.instruction)
    plan = reasoner.plan(t.instruction, perc, {"holding_pen": False}, mission_phase="explore")
    rows.append({
        "task": t.id,
        "instruction": t.instruction,
        "caption": perc.caption,
        "objects": ",".join(perc.objects_of_interest),
        "intent": plan.intent,
        "risk": plan.risk,
        "latency_ms": (time.perf_counter() - t0) * 1000,
    })
pd.DataFrame(rows)